# Brazil: análise temporal e comparação de modelos
Grupo 4 — especialização em MLP Regressor.
Previsão de gols na próxima partida, com SARIMAX, Holt-Winters e Random Forest
como referências. O teste final é mantido fora da seleção de configurações.

In [1]:
from pathlib import Path
import base64
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import warnings
from itertools import product

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from threadpoolctl import threadpool_limits

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "datasets/brazil.csv").is_file())
OUT = ROOT / "outputs_brazil_comparacao"
OUT.mkdir(exist_ok=True)
FIG = OUT / "figuras"
FIG.mkdir(exist_ok=True)
SEED = 42
CV_SPLITS = 4
PERIODS = [4, 6, 8, 10, 12, 20, 30, 50]
LAGS = [1, 2, 3, 5, 10]
WINDOWS = [3, 5, 10, 20]
TARGET = "goals_brazil"
COLORS = {"MLP Regressor": "#533afd", "Random Forest": "#ea2261",
          "SARIMAX": "#007fa3", "Holt-Winters": "#9b6829"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .15, "figure.dpi": 120})
threadpool_limits(limits=2)


def log(message):
    stamp = pd.Timestamp.now(tz="America/Sao_Paulo").isoformat()
    with (OUT / "progress.log").open("a", encoding="utf-8") as f:
        f.write(f"{stamp} | {message}\n")
    print(message, flush=True)


def save_json(name, obj):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2,
                                      default=str, allow_nan=False), encoding="utf-8")


def savefig(name, fig):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight", dpi=140)
    plt.close(fig)


def preprocessor(num, cat, sarimax=False):
    return ColumnTransformer([
        ("numeric", StandardScaler(), num),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                      drop="first" if sarimax else None,
                                      min_frequency=None if sarimax else 5), cat)],
        verbose_feature_names_out=True)


log("Início da análise reproduzível do Brazil")

Início da análise reproduzível do Brazil


## 1. Qualidade dos dados e definição do problema
A versão original do CSV é preservada e identificada por SHA-256.
Sem horários para ordenar partidas no mesmo dia, mantém-se o primeiro registro
de cada data conforme a ordem original. As exclusões ficam em um arquivo de auditoria.
O indicador `is_home` descreve a ordem nominal do confronto, não o estádio real.
O placar principal exclui a disputa por pênaltis. Os pontos históricos usam W/D/L
registrados na base; por isso, podem refletir o resultado após pênaltis.

In [2]:
source = pd.read_csv(ROOT / "datasets/brazil.csv")
raw = source.copy()
raw["original_row"] = np.arange(len(raw)) + 2
raw["date"] = pd.to_datetime(raw["Date"], format="%d %b %Y", errors="coerce")
raw = raw.sort_values("date", kind="stable")
same_day = raw.duplicated("date", keep="first") & raw["date"].notna()
excluded = raw.loc[same_day].assign(reason="Data repetida sem horário: primeiro registro mantido")
raw = raw.loc[~same_day].copy()
score = raw["Score"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)")
raw["home_goals"] = pd.to_numeric(score[0], errors="coerce")
raw["away_goals"] = pd.to_numeric(score[1], errors="coerce")
raw["is_home"] = raw["Match"].str.startswith("Brazil v ").astype(int)
raw["opponent"] = np.where(raw["is_home"].eq(1),
    raw["Match"].str.replace(r"^Brazil v ", "", regex=True),
    raw["Match"].str.replace(r" v Brazil$", "", regex=True))
raw[TARGET] = np.where(raw["is_home"].eq(1), raw["home_goals"], raw["away_goals"])
raw["goals_against"] = np.where(raw["is_home"].eq(1), raw["away_goals"], raw["home_goals"])
raw["points"] = raw["Result"].map({"W": 3, "D": 1, "L": 0})
required = ["date", "Competition", "opponent", TARGET, "goals_against", "points"]
invalid = raw[required].isna().any(axis=1)
excluded = pd.concat([excluded, raw.loc[invalid].assign(reason="Campo obrigatório ausente ou inválido")])
excluded.to_csv(OUT / "registros_excluidos.csv", index=False)
raw = raw.loc[~invalid].reset_index(drop=True)
assert raw["date"].is_monotonic_increasing and raw["date"].is_unique
gaps = raw["date"].diff().dt.days.dropna()
q1, q3 = raw[TARGET].quantile([.25, .75])
extreme = raw[TARGET] > q3 + 1.5 * (q3 - q1)
quality = {
    "sha256": hashlib.sha256((ROOT / "datasets/brazil.csv").read_bytes()).hexdigest(),
    "raw_rows": len(source), "clean_rows": len(raw), "same_day_removed": int(same_day.sum()),
    "invalid_removed": int(invalid.sum()), "exact_duplicates": int(source.duplicated().sum()),
    "missing_original": source.isna().sum().to_dict(),
    "start": raw.date.min().strftime("%d/%m/%Y"), "end": raw.date.max().strftime("%d/%m/%Y"),
    "competitions": int(raw.Competition.nunique()), "opponents": int(raw.opponent.nunique()),
    "mean_goals": float(raw[TARGET].mean()), "median_goals": float(raw[TARGET].median()),
    "std_goals": float(raw[TARGET].std()), "min_goals": float(raw[TARGET].min()),
    "max_goals": float(raw[TARGET].max()), "nominal_home_fraction": float(raw.is_home.mean()),
    "gap_median": float(gaps.median()), "gap_mean": float(gaps.mean()), "gap_max": int(gaps.max()),
    "extreme_count": int(extreme.sum()), "extreme_threshold": float(q3 + 1.5 * (q3-q1))}
raw.loc[extreme, ["date", "Match", "Score", TARGET]].to_csv(OUT / "placares_atipicos.csv", index=False)
raw.to_csv(OUT / "dados_limpos.csv", index=False)
save_json("qualidade_dados.json", quality)
display(pd.DataFrame([quality]).drop(columns=["missing_original"]))
fig, axes = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(raw.date, raw[TARGET], color="#533afd", linewidth=.7, alpha=.65)
axes[0].plot(raw.date, raw[TARGET].rolling(30).mean(), color="#ea2261", label="Média de 30 partidas")
axes[0].set(ylabel="Gols por partida", title="Gols da Seleção Brasileira ao longo do tempo")
axes[0].legend()
axes[1].hist(raw[TARGET], bins=np.arange(-.5, raw[TARGET].max()+1.5), color="#533afd")
axes[1].set(xlabel="Gols por partida", ylabel="Número de partidas")
savefig("serie_e_distribuicao", fig)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
raw.groupby("is_home")[TARGET].mean().rename(index={0: "Visitante nominal", 1: "Mandante nominal"}).plot.bar(ax=axes[0], color=["#007fa3", "#533afd"], rot=15)
axes[0].set(ylabel="Média de gols", xlabel="", title="Ordem do confronto")
raw.Competition.value_counts().head(6).sort_values().plot.barh(ax=axes[1], color="#533afd")
axes[1].set(title="Competições frequentes", xlabel="Partidas")
axes[2].hist(gaps.clip(upper=180), bins=25, color="#007fa3")
axes[2].set(title="Intervalo entre partidas", xlabel="Dias (cauda agrupada em 180)", ylabel="Frequência")
savefig("variaveis_externas", fig)

,sha256,raw_rows,clean_rows,same_day_removed,invalid_removed,exact_duplicates,start,end,competitions,opponents,...,median_goals,std_goals,min_goals,max_goals,nominal_home_fraction,gap_median,gap_mean,gap_max,extreme_count,extreme_threshold
0,95c9561f80ec6c0a353d97849d922aa305a55f1445bf77...,1034,1030,3,1,0,20/09/1914,09/12/2022,44,96,...,2.0,1.741079,0.0,10.0,0.585437,7.0,38.413022,1662,24,6.0


## 2. Engenharia de atributos e origens de previsão
Cada observação corresponde a uma partida; a frequência do índice de eventos
é regular (um jogo por passo), enquanto os intervalos de calendário variam.
A origem operacional é o fim do dia anterior ao jogo. Pressupõe-se que data,
adversário, competição e ordem do confronto estejam conhecidos nessa origem.
Os resultados da partida-alvo nunca entram nos atributos dessa partida.
As primeiras 20 observações, sem histórico suficiente, são retiradas para todos
os modelos. Não se usa preenchimento com dados posteriores.

In [3]:
data = raw.copy()
data["days_since_last_match"] = data.date.diff().dt.days
data["log_days_since_last_match"] = np.log1p(data.days_since_last_match)
for lag in LAGS:
    for prefix, col in [("goals", TARGET), ("conceded", "goals_against"), ("points", "points")]:
        data[f"{prefix}_lag_{lag}"] = data[col].shift(lag)
for w in WINDOWS:
    data[f"goals_roll_mean_{w}"] = data[TARGET].shift(1).rolling(w).mean()
    data[f"goals_roll_std_{w}"] = data[TARGET].shift(1).rolling(w).std()
    data[f"conceded_roll_mean_{w}"] = data.goals_against.shift(1).rolling(w).mean()
    data[f"points_roll_mean_{w}"] = data.points.shift(1).rolling(w).mean()
data["month_sin"] = np.sin(2*np.pi*data.date.dt.month/12)
data["month_cos"] = np.cos(2*np.pi*data.date.dt.month/12)
data["doy_sin"] = np.sin(2*np.pi*data.date.dt.dayofyear/365.25)
data["doy_cos"] = np.cos(2*np.pi*data.date.dt.dayofyear/365.25)
data["year"] = data.date.dt.year
num = ["is_home", "log_days_since_last_match", "month_sin", "month_cos", "doy_sin", "doy_cos", "year"]
num += [f"{prefix}_lag_{lag}" for lag in LAGS for prefix in ["goals", "conceded", "points"]]
num += [f"{prefix}_{w}" for w in WINDOWS for prefix in ["goals_roll_mean", "goals_roll_std", "conceded_roll_mean", "points_roll_mean"]]
cat = ["opponent", "Competition"]
features = num + cat
data = data.dropna(subset=features+[TARGET]).reset_index(drop=True)
cut = int(len(data)*.8)
train, test = data.iloc[:cut].copy(), data.iloc[cut:].copy()
X_train, X_test = train[features], test[features]
y_train, y_test = train[TARGET].to_numpy(float), test[TARGET].to_numpy(float)
origin = test.date - pd.Timedelta(seconds=1)
cv = list(TimeSeriesSplit(n_splits=CV_SPLITS).split(X_train))
for a, b in cv:
    assert a.max() < b.min() and train.date.iloc[a.max()] < train.date.iloc[b.min()]
assert train.date.max() < test.date.min()
assert not set([TARGET, "goals_against", "points", "Score", "Result"]).intersection(features)
assert np.allclose(data.goals_lag_1, raw[TARGET].shift(1).iloc[len(raw)-len(data):])
protocol = {"horizon": "1 partida", "origin": "23:59:59 do dia anterior à partida",
            "frequency": "Índice regular de eventos: uma partida por passo",
            "initial_train": len(train), "test_origins": len(test), "warmup_removed": len(raw)-len(data),
            "train_start": train.date.min().strftime("%d/%m/%Y"), "train_end": train.date.max().strftime("%d/%m/%Y"),
            "test_start": test.date.min().strftime("%d/%m/%Y"), "test_end": test.date.max().strftime("%d/%m/%Y"),
            "numeric_features": num, "categorical_features": cat, "random_state": SEED,
            "cv": [{"fold": i+1, "train_n": len(a), "validation_n": len(b),
                    "train_end": train.date.iloc[a[-1]].strftime("%d/%m/%Y"),
                    "validation_start": train.date.iloc[b[0]].strftime("%d/%m/%Y"),
                    "validation_end": train.date.iloc[b[-1]].strftime("%d/%m/%Y")} for i,(a,b) in enumerate(cv)]}
save_json("protocolo.json", protocol)
data.to_csv(OUT / "dados_modelagem.csv", index=False)
display(pd.DataFrame(protocol["cv"]))

,fold,train_n,validation_n,train_end,validation_start,validation_end
0,1,164,161,19/06/1958,24/06/1958,06/06/1973
1,2,325,161,06/06/1973,09/06/1973,12/12/1987
2,3,486,161,12/12/1987,07/07/1988,10/06/1998
3,4,647,161,10/06/1998,16/06/1998,31/05/2008


## 3. STL e força da sazonalidade
Os períodos candidatos são examinados somente no treino. A STL não é usada
como atributo preditor: sua suavização pode usar pontos posteriores dentro do
segmento de análise. A força sazonal é Fs=max(0,1-Var(R)/Var(S+R)).
A decomposição robusta mantém placares extremos no componente residual.

In [4]:
decompositions = {}
stl_rows = []
for m in PERIODS:
    fit = STL(y_train, period=m, robust=True).fit()
    decompositions[m] = fit
    fs = max(0., 1.-np.var(fit.resid)/np.var(fit.seasonal+fit.resid))
    ft = max(0., 1.-np.var(fit.resid)/np.var(fit.trend+fit.resid))
    stl_rows.append({"period": m, "seasonal_strength": fs, "trend_strength": ft})
stl_table = pd.DataFrame(stl_rows).sort_values("seasonal_strength", ascending=False)
stl_table.to_csv(OUT / "forca_sazonal.csv", index=False)
STL_PERIOD = int(stl_table.iloc[0].period)
stl_fit = decompositions[STL_PERIOD]
pd.DataFrame({"date": train.date, "observed": y_train, "trend": stl_fit.trend,
              "seasonal": stl_fit.seasonal, "residual": stl_fit.resid}).to_csv(OUT / "decomposicao_stl.csv", index=False)
fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for ax, values, label in zip(axes, [y_train, stl_fit.trend, stl_fit.seasonal, stl_fit.resid],
                             ["Observado", "Tendência", "Sazonal", "Resíduo"]):
    ax.plot(train.date, values, color="#533afd", linewidth=.9)
    ax.set_ylabel(label)
axes[0].set_title(f"STL robusta no treino — período de {STL_PERIOD} partidas")
savefig("stl", fig)
trend_segments = []
for ids in np.array_split(np.arange(len(train)), 6):
    first, last = float(np.median(stl_fit.trend[ids[:min(10,len(ids))]])), float(np.median(stl_fit.trend[ids[-min(10,len(ids)):]]))
    delta = last-first
    trend_segments.append({"start": train.date.iloc[ids[0]].strftime("%d/%m/%Y"),
        "end": train.date.iloc[ids[-1]].strftime("%d/%m/%Y"), "initial_level": first,
        "final_level": last, "change": delta, "direction": "crescimento" if delta>.1 else "queda" if delta<-.1 else "estabilidade"})
pd.DataFrame(trend_segments).to_csv(OUT / "interpretacao_tendencia.csv", index=False)
display(stl_table)
display(pd.DataFrame(trend_segments))
log(f"STL concluída: maior força sazonal {stl_table.iloc[0].seasonal_strength:.4f}, m={STL_PERIOD}")

,period,seasonal_strength,trend_strength
6,30,0.244277,0.056673
7,50,0.200106,0.037617
3,10,0.195794,0.121422
5,20,0.183066,0.087733
1,6,0.182973,0.151103
2,8,0.169300,0.131156
4,12,0.131342,0.073121
0,4,0.106328,0.150309


,start,end,initial_level,final_level,change,direction
0,02/10/1921,12/06/1956,2.226787,1.619552,-0.607235,queda
1,17/06/1956,20/06/1968,1.686405,2.017639,0.331234,crescimento
2,23/06/1968,29/06/1980,2.118508,2.175845,0.057336,estabilidade
3,27/08/1980,18/12/1991,2.176835,1.659357,-0.517478,queda
4,26/02/1992,07/09/1999,1.797921,2.148637,0.350716,crescimento
5,09/10/1999,31/05/2008,1.870408,2.326916,0.456507,crescimento


STL concluída: maior força sazonal 0.2443, m=30


## 4. Otimização temporal: Random Forest e MLP
Quatro blocos expansivos do treino, sem embaralhamento. Em cada bloco,
padronização e codificação categórica são ajustadas exclusivamente no passado.
Categorias com menos de cinco ocorrências no treino são agrupadas pelo encoder.
A seleção minimiza o MAE médio. A tabela registra todos os candidatos, duração,
MAEs por divisão e avisos de convergência. O teste não participa da busca.

In [5]:
RF_SPACE = {"n_estimators": [150, 300, 500], "max_depth": [None, 6, 10],
            "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 3, 6],
            "max_features": ["sqrt", .5, 1.]}
MLP_SPACE = {"hidden_layer_sizes": [(16,), (32,), (64,), (32,16)],
             "activation": ["relu", "tanh"], "alpha": [.01, .1, 1., 10., 100.],
             "learning_rate_init": [.001, .003], "solver": ["adam", "lbfgs"]}
search_spaces = {"Random Forest": RF_SPACE, "MLP Regressor": MLP_SPACE}
selected = {}
search_times = {}


def tabular_model(name, params, seed=SEED):
    if name == "Random Forest":
        estimator = RandomForestRegressor(**params, random_state=seed, n_jobs=2)
    else:
        estimator = MLPRegressor(**params, random_state=seed, max_iter=1000,
                                 max_fun=20000, early_stopping=False, n_iter_no_change=30)
    return Pipeline([("preprocess", preprocessor(num,cat)), ("model", estimator)])


def tune_tabular(name, count):
    slug = "mlp" if name=="MLP Regressor" else "random_forest"
    cache = OUT / f"busca_{slug}.csv"
    candidates = list(ParameterSampler(search_spaces[name], n_iter=count, random_state=SEED))
    if os.environ.get("BRAZIL_RESUME") == "1" and cache.is_file():
        table = pd.read_csv(cache).sort_values(["cv_mae","candidate"]).reset_index(drop=True)
        expected = {i+1: json.dumps(params, sort_keys=True) for i,params in enumerate(candidates)}
        recorded = {int(row.candidate): json.dumps(json.loads(row.parameters),sort_keys=True)
                    for row in table.itertuples()}
        if recorded == expected:
            best = json.loads(table.iloc[0].parameters)
            if "hidden_layer_sizes" in best: best["hidden_layer_sizes"] = tuple(best["hidden_layer_sizes"])
            selected[name] = {"parameters":best,"cv_mae":float(table.iloc[0].cv_mae),
                "cv_std":float(table.iloc[0].cv_std),"candidates":count,
                "convergence_warnings":int(table.iloc[0].convergence_warnings)}
            search_times[name] = float(table.seconds.sum())
            save_json("configuracoes_selecionadas.json",selected)
            log(f"{name}: resultados da busca anterior conferidos e reutilizados")
            return table
    rows = []
    started = time.perf_counter()
    for i,params in enumerate(ParameterSampler(search_spaces[name], n_iter=count, random_state=SEED)):
        t0 = time.perf_counter()
        scores, train_scores, warning_count = [], [], 0
        for a,b in cv:
            model = tabular_model(name,params)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                model.fit(X_train.iloc[a],y_train[a])
            warning_count += sum(issubclass(w.category, ConvergenceWarning) for w in caught)
            scores.append(mean_absolute_error(y_train[b], model.predict(X_train.iloc[b])))
            train_scores.append(mean_absolute_error(y_train[a], model.predict(X_train.iloc[a])))
        rows.append({"candidate":i+1,"parameters":json.dumps(params,default=str),
            "cv_mae":float(np.mean(scores)), "cv_std":float(np.std(scores)),
            "training_mae":float(np.mean(train_scores)),"convergence_warnings":warning_count,
            "seconds":time.perf_counter()-t0, **{f"fold_{j+1}":v for j,v in enumerate(scores)}})
        log(f"{name}: candidato {i+1}/{count}, MAE CV={np.mean(scores):.4f}")
    table = pd.DataFrame(rows).sort_values(["cv_mae","candidate"]).reset_index(drop=True)
    slug = "mlp" if name=="MLP Regressor" else "random_forest"
    table.to_csv(OUT / f"busca_{slug}.csv",index=False)
    best = json.loads(table.iloc[0].parameters)
    if "hidden_layer_sizes" in best: best["hidden_layer_sizes"] = tuple(best["hidden_layer_sizes"])
    selected[name] = {"parameters":best,"cv_mae":float(table.iloc[0].cv_mae),
                      "cv_std":float(table.iloc[0].cv_std),"candidates":count,
                      "convergence_warnings":int(table.iloc[0].convergence_warnings)}
    search_times[name] = time.perf_counter()-started
    save_json("configuracoes_selecionadas.json",selected)
    return table


rf_search = tune_tabular("Random Forest", 18)
display(rf_search.head(5))

Random Forest: resultados da busca anterior conferidos e reutilizados


,candidate,parameters,cv_mae,cv_std,training_mae,convergence_warnings,seconds,fold_1,fold_2,fold_3,fold_4
0,18,"{""n_estimators"": 150, ""min_samples_split"": 10,...",1.309413,0.097685,0.967027,0,2.206583,1.329326,1.352408,1.147679,1.408241
1,6,"{""n_estimators"": 300, ""min_samples_split"": 5, ...",1.314388,0.098798,0.944636,0,4.402290,1.332664,1.356849,1.151612,1.416424
2,11,"{""n_estimators"": 150, ""min_samples_split"": 2, ...",1.314836,0.093034,0.995019,0,3.515509,1.367140,1.336377,1.158131,1.397693
3,7,"{""n_estimators"": 500, ""min_samples_split"": 10,...",1.315343,0.091960,1.007236,0,7.847725,1.371337,1.338784,1.159463,1.391786
4,12,"{""n_estimators"": 500, ""min_samples_split"": 2, ...",1.315343,0.091960,1.007236,0,8.163018,1.371337,1.338784,1.159463,1.391786


In [6]:
mlp_search = tune_tabular("MLP Regressor", 24)
display(mlp_search.head(8))

MLP Regressor: resultados da busca anterior conferidos e reutilizados

,candidate,parameters,cv_mae,cv_std,training_mae,convergence_warnings,seconds,fold_1,fold_2,fold_3,fold_4
0,18,"{""solver"": ""lbfgs"", ""learning_rate_init"": 0.00...",1.303732,0.095145,1.331084,0,1.270369,1.330453,1.333408,1.147032,1.404034
1,21,"{""solver"": ""lbfgs"", ""learning_rate_init"": 0.00...",1.310235,0.094480,1.299937,3,8.688046,1.335272,1.336810,1.155898,1.412961
2,14,"{""solver"": ""adam"", ""learning_rate_init"": 0.003...",1.344183,0.079927,1.381888,0,3.849963,1.346839,1.351878,1.226240,1.451776
3,17,"{""solver"": ""adam"", ""learning_rate_init"": 0.001...",1.396724,0.183587,0.908890,3,15.881974,1.633965,1.426610,1.118313,1.408008
4,4,"{""solver"": ""lbfgs"", ""learning_rate_init"": 0.00...",1.498385,0.142850,0.354978,4,5.482963,1.708977,1.394465,1.343100,1.546997
5,8,"{""solver"": ""lbfgs"", ""learning_rate_init"": 0.00...",1.553816,0.168400,0.420270,4,3.655710,1.758966,1.470285,1.324224,1.661789
6,10,"{""solver"": ""adam"", ""learning_rate_init"": 0.003...",1.588103,0.229116,0.564426,0,9.290112,1.940118,1.633390,1.339021,1.439882
7,15,"{""solver"": ""adam"", ""learning_rate_init"": 0.001...",1.592999,0.150176,0.057864,1,10.324549,1.759170,1.573037,1.359408,1.680381


## 5. Holt-Winters: referência univariada
Investiga tendência, amortecimento e sazonalidade aditiva; componentes
multiplicativos são excluídos porque a série contém zeros. Os coeficientes de
suavização são estimados no treino por otimização com limites explícitos.
A estrutura é selecionada pelo menor BIC, sem usar o teste final.

In [7]:
HW_BOUNDS = {"smoothing_level": (.0001,.9999), "smoothing_trend": (0.,.9999),
             "smoothing_seasonal": (0.,.9999), "damping_trend": (.8,.995)}
hw_configs = []
for trend, damped in [(None,False),("add",False),("add",True)]:
    hw_configs.append({"trend":trend,"damped_trend":damped,"seasonal":None,"seasonal_periods":None})
    for m in sorted(set([4,12,30,STL_PERIOD])):
        hw_configs.append({"trend":trend,"damped_trend":damped,"seasonal":"add","seasonal_periods":m})


def fit_hw(y, config):
    return ExponentialSmoothing(y, **config, initialization_method="estimated",
                                bounds=HW_BOUNDS).fit(optimized=True, use_brute=True)


t0 = time.perf_counter()
hw_rows, hw_fits = [], {}
for i,config in enumerate(hw_configs):
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fit = fit_hw(y_train,config)
    hw_fits[i] = fit
    hw_rows.append({"candidate":i,"parameters":json.dumps(config),"AIC":fit.aic,"BIC":fit.bic,
                   "SSE":fit.sse,"seconds":time.perf_counter()-started,"warnings":len(caught)})
hw_search = pd.DataFrame(hw_rows).sort_values("BIC")
hw_search.to_csv(OUT / "busca_holt_winters.csv",index=False)
best_hw_id = int(hw_search.iloc[0].candidate)
HW_CONFIG = hw_configs[best_hw_id]
hw_initial = hw_fits[best_hw_id]
smooth_params = {k:float(v) for k,v in hw_initial.params.items()
                 if k in ["smoothing_level","smoothing_trend","smoothing_seasonal","damping_trend"] and np.isfinite(v)}
selected["Holt-Winters"] = {"parameters":HW_CONFIG,"smoothing_parameters":smooth_params,
                           "BIC":float(hw_initial.bic),"AIC":float(hw_initial.aic),"candidates":len(hw_configs)}
search_times["Holt-Winters"] = time.perf_counter()-t0
display(hw_search.head(6))
log(f"Holt-Winters selecionado: {HW_CONFIG}")

,candidate,parameters,AIC,BIC,SSE,seconds,warnings
0,0,"{""trend"": null, ""damped_trend"": false, ""season...",929.188008,938.577132,2539.180631,0.021842,0
4,4,"{""trend"": ""add"", ""damped_trend"": false, ""seaso...",933.045138,951.823387,2538.731696,0.150624,0
8,8,"{""trend"": ""add"", ""damped_trend"": true, ""season...",929.151612,952.624423,2520.281638,0.057825,0
1,1,"{""trend"": null, ""damped_trend"": false, ""season...",936.029496,964.196868,2535.542558,0.097744,0
5,5,"{""trend"": ""add"", ""damped_trend"": false, ""seaso...",939.887968,977.444465,2535.098475,0.348176,0
9,9,"{""trend"": ""add"", ""damped_trend"": true, ""season...",936.008159,978.259218,2516.717546,0.477709,0


Holt-Winters selecionado: {'trend': None, 'damped_trend': False, 'seasonal': None, 'seasonal_periods': None}


## 6. SARIMAX com variáveis externas
As competições são agrupadas por regra fixa, independente de gols e do teste,
reduzindo categorias raras. Mandante nominal, intervalo entre jogos, calendário,
ano e competição são conhecidos na origem assumida. O encoder usa uma categoria
de referência para evitar redundância com a constante.
Ordens e períodos sazonais são escolhidos por BIC no treino; ajustes sem
convergência ou BIC finito não concorrem. Todos os candidatos são registrados.

In [8]:
sar_num = ["is_home","log_days_since_last_match","month_sin","month_cos","year"]
sar_cat = ["competition_group"]


def sar_data(frame):
    result = frame.copy()
    result["competition_group"] = result.Competition.where(result.Competition.isin([
        "International Friendly", "FIFA World Cup", "Copa America", "FIFA Confederations Cup"]),"Outras competições")
    return result[sar_num+sar_cat]


sar_pre = preprocessor(sar_num,sar_cat,sarimax=True)
sar_X_train = sar_pre.fit_transform(sar_data(train))
sar_X_test = sar_pre.transform(sar_data(test))
sar_configs = [((p,d,q),(0,0,0,0)) for p,d,q in product([0,1,2],[0,1],[0,1,2])]
for m in sorted(set([4,12,30,STL_PERIOD])):
    for order, seasonal in [((1,0,0),(1,0,0,m)),((1,0,1),(1,0,0,m)),
                             ((0,1,1),(0,1,1,m)),((1,0,1),(0,0,1,m))]:
        sar_configs.append((order,seasonal))


def fit_sar(y,X,order,seasonal,maxiter=300):
    return SARIMAX(y,exog=X,order=order,seasonal_order=seasonal,trend="c",
                   enforce_stationarity=True,enforce_invertibility=True).fit(disp=False,maxiter=maxiter)


t0 = time.perf_counter()
sar_rows, sar_fits = [], {}
for i,(order,seasonal) in enumerate(sar_configs):
    started = time.perf_counter()
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fit = fit_sar(y_train,sar_X_train,order,seasonal)
        converged = bool(fit.mle_retvals.get("converged",False))
        if converged and np.isfinite(fit.bic): sar_fits[i] = fit
        sar_rows.append({"candidate":i,"order":str(order),"seasonal_order":str(seasonal),
            "AIC":float(fit.aic),"BIC":float(fit.bic),"converged":converged,
            "seconds":time.perf_counter()-started,"warnings":len(caught),"error":""})
        log(f"SARIMAX {i+1}/{len(sar_configs)}: BIC={fit.bic:.2f}, convergiu={converged}")
    except (ValueError, np.linalg.LinAlgError) as error:
        sar_rows.append({"candidate":i,"order":str(order),"seasonal_order":str(seasonal),
                         "converged":False,"error":str(error),"seconds":time.perf_counter()-started})
sar_search = pd.DataFrame(sar_rows).sort_values("BIC",na_position="last")
sar_search.to_csv(OUT / "busca_sarimax.csv",index=False)
eligible = sar_search[sar_search.candidate.isin(sar_fits)]
if eligible.empty: raise RuntimeError("Nenhum SARIMAX convergiu: verifique busca_sarimax.csv")
best_sar_id = int(eligible.iloc[0].candidate)
SAR_ORDER, SAR_SEASONAL = sar_configs[best_sar_id]
sar_initial = sar_fits[best_sar_id]
selected["SARIMAX"] = {"parameters":{"order":SAR_ORDER,"seasonal_order":SAR_SEASONAL},
    "BIC":float(sar_initial.bic),"AIC":float(sar_initial.aic),"candidates":len(sar_configs),
    "converged_candidates":len(sar_fits),"exogenous_columns":sar_num+sar_cat}
search_times["SARIMAX"] = time.perf_counter()-t0
save_json("configuracoes_selecionadas.json",selected)
save_json("espacos_de_busca.json",{"tabular":search_spaces,"folds":CV_SPLITS,
    "holt_winters":hw_configs,"smoothing_bounds":HW_BOUNDS,"sarimax":sar_configs})
display(eligible.head(8))

SARIMAX 1/30: BIC=3240.44, convergiu=True


SARIMAX 2/30: BIC=3246.54, convergiu=True


SARIMAX 3/30: BIC=3252.97, convergiu=True


SARIMAX 4/30: BIC=3762.98, convergiu=True


SARIMAX 5/30: BIC=3245.09, convergiu=True


...

SARIMAX 27/30: BIC=3247.43, convergiu=True


SARIMAX 28/30: BIC=3247.31, convergiu=True


SARIMAX 29/30: BIC=3221.66, convergiu=True


SARIMAX 30/30: BIC=3247.40, convergiu=True


,candidate,order,seasonal_order,AIC,BIC,converged,seconds,warnings,error
28,28,"(0, 1, 1)","(0, 1, 1, 30)",3161.135032,3221.655756,True,184.764225,0,
0,0,"(0, 0, 0)","(0, 0, 0, 0)",3188.798535,3240.438718,True,0.738737,0,
4,4,"(0, 1, 1)","(0, 0, 0, 0)",3188.769822,3245.089706,True,8.144692,0,
7,7,"(1, 0, 1)","(0, 0, 0, 0)",3185.425533,3246.454840,True,3.853491,0,
6,6,"(1, 0, 0)","(0, 0, 0, 0)",3190.178873,3246.513617,True,0.699789,0,
1,1,"(0, 0, 1)","(0, 0, 0, 0)",3190.203259,3246.538004,True,1.110160,0,
27,27,"(1, 0, 1)","(1, 0, 0, 30)",3181.582902,3247.306771,True,19.193788,0,
29,29,"(1, 0, 1)","(0, 0, 1, 30)",3181.678351,3247.402220,True,14.744131,0,


## 7. Walk-forward: previsões fora da amostra
Todos os modelos usam as mesmas origens e horizonte de uma partida.
Random Forest e MLP são reajustados com janela expansiva; o pré-processamento
também é reajustado usando apenas o histórico disponível. O SARIMAX atualiza
seu estado com o resultado após cada previsão, mantendo os parâmetros estimados
no treino. Holt-Winters atualiza nível/tendência/sazonalidade pelos coeficientes
de suavização estimados no treino, sem nova otimização no teste.
Não se arredondam nem truncam previsões: o protocolo mede regressão contínua.

In [9]:
predictions = {}
initial_models = {}
timing = {}
for name in ["Random Forest","MLP Regressor"]:
    initial_models[name] = tabular_model(name, selected[name]["parameters"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        initial_models[name].fit(X_train,y_train)
    model_dir = OUT / ("mlp" if name=="MLP Regressor" else "random_forest")
    model_dir.mkdir(exist_ok=True)
    joblib.dump(initial_models[name],model_dir / "modelo_treino_inicial.joblib")
    pred, elapsed, iterations, conv = [], [], [], []
    for i in range(len(test)):
        started = time.perf_counter()
        history = data.iloc[:cut+i]
        model = tabular_model(name,selected[name]["parameters"])
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(history[features],history[TARGET].to_numpy(float))
        pred.append(float(model.predict(test.iloc[[i]][features])[0]))
        elapsed.append(time.perf_counter()-started)
        iterations.append(getattr(model.named_steps["model"],"n_iter_",0))
        conv.append(sum(issubclass(w.category,ConvergenceWarning) for w in caught))
        if i%25==0 or i==len(test)-1: log(f"Walk-forward {name}: {i+1}/{len(test)}")
    predictions[name] = np.array(pred)
    timing[name] = {"search_seconds":search_times[name],"forecast_seconds":float(sum(elapsed)),
                    "convergence_warnings":int(sum(conv)),"mean_fit_iterations":float(np.mean(iterations))}
    pd.DataFrame({"target_date":test.date,"origin_date":origin,"real":y_test,
        "prediction":pred,"residual":y_test-pred,"absolute_error":np.abs(y_test-pred),
        "seconds":elapsed,"fit_iterations":iterations,"convergence_warnings":conv}).to_csv(model_dir / "walk_forward_predictions.csv",index=False)
    joblib.dump(model,model_dir / "modelo_ultima_origem.joblib")
    save_json(f"{model_dir.name}/config.json",selected[name])

Walk-forward Random Forest: 1/202


Walk-forward Random Forest: 26/202


Walk-forward Random Forest: 51/202


Walk-forward Random Forest: 76/202


Walk-forward Random Forest: 101/202


...

Walk-forward MLP Regressor: 126/202


Walk-forward MLP Regressor: 151/202


Walk-forward MLP Regressor: 176/202


Walk-forward MLP Regressor: 201/202


Walk-forward MLP Regressor: 202/202


In [10]:
started = time.perf_counter()
sar_state = sar_initial
sar_pred = []
for i in range(len(test)):
    sar_pred.append(float(np.ravel(sar_state.get_forecast(steps=1,exog=sar_X_test[i:i+1]).predicted_mean)[0]))
    sar_state = sar_state.append(y_test[i:i+1],exog=sar_X_test[i:i+1],refit=False)
    if i%50==0 or i==len(test)-1: log(f"Walk-forward SARIMAX: {i+1}/{len(test)}")
predictions["SARIMAX"] = np.array(sar_pred)
timing["SARIMAX"] = {"search_seconds":search_times["SARIMAX"],"forecast_seconds":time.perf_counter()-started,"convergence_warnings":0}
(OUT / "sarimax").mkdir(exist_ok=True)
sar_initial.save(OUT / "sarimax/modelo_treino_inicial.pkl")
sar_state.save(OUT / "sarimax/modelo_atualizado.pkl")
joblib.dump(sar_pre,OUT / "sarimax/preprocessador.joblib")

Walk-forward SARIMAX: 1/202


Walk-forward SARIMAX: 51/202


Walk-forward SARIMAX: 101/202


Walk-forward SARIMAX: 151/202


Walk-forward SARIMAX: 201/202


Walk-forward SARIMAX: 202/202


['C:\\Users\\RafaellaAntunes-ieg\\OneDrive - Instituto J&F\\GERMINARE TECH 2025\\3° Ano\\Serie temporal\\trabalhoseries_redeneural\\outputs_brazil_comparacao\\sarimax\\preprocessador.joblib']

In [11]:
started = time.perf_counter()
alpha = float(hw_initial.params["smoothing_level"])
beta = float(hw_initial.params["smoothing_trend"]) if HW_CONFIG["trend"] else 0.
gamma = float(hw_initial.params["smoothing_seasonal"]) if HW_CONFIG["seasonal"] else 0.
phi = float(hw_initial.params["damping_trend"]) if HW_CONFIG["damped_trend"] else 1.
level = float(hw_initial.level[-1])
slope = float(hw_initial.trend[-1]) if HW_CONFIG["trend"] else 0.
m = HW_CONFIG["seasonal_periods"] or 1
seasons = list(map(float,hw_initial.season[-m:])) if HW_CONFIG["seasonal"] else [0.]
# A primeira previsão recursiva deve coincidir com a implementação de referência.
assert np.isclose(level+phi*slope+seasons[0],np.ravel(hw_initial.forecast(1))[0])
hw_pred = []
for i,y in enumerate(y_test):
    season = seasons[i%m]
    projected = level+phi*slope
    hw_pred.append(projected+season)
    new_level = alpha*(y-season)+(1-alpha)*projected
    new_slope = beta*(new_level-level)+(1-beta)*phi*slope if HW_CONFIG["trend"] else 0.
    if HW_CONFIG["seasonal"]:
        seasons[i%m] = gamma*(y-projected)+(1-gamma)*season
    level, slope = new_level,new_slope
predictions["Holt-Winters"] = np.array(hw_pred)
timing["Holt-Winters"] = {"search_seconds":search_times["Holt-Winters"],"forecast_seconds":time.perf_counter()-started,"convergence_warnings":0}
(OUT / "holt_winters").mkdir(exist_ok=True)
joblib.dump(hw_initial,OUT / "holt_winters/modelo_treino_inicial.joblib")
save_json("holt_winters/estado_final.json",{"level":level,"trend":slope,"seasonal":seasons,"config":selected["Holt-Winters"]})
save_json("tempos_execucao.json",timing)

## 8. MAE, incerteza da comparação e subperíodos
A tabela principal contém apenas os quatro modelos exigidos. Uma referência
ingênua de mediana histórica é registrada separadamente para contextualizar o
MAE. Intervalos de diferença de MAE usam bootstrap de blocos consecutivos de
dez jogos, preservando parte da dependência temporal; são exploratórios.

In [12]:
comparison_rows = []
combined = pd.DataFrame({"target_date":test.date,"origin_date":origin,"real":y_test})
for name,pred in predictions.items():
    assert len(pred)==len(test) and np.isfinite(pred).all()
    residual = y_test-pred
    comparison_rows.append({"modelo":name,"MAE_teste":mean_absolute_error(y_test,pred),
        "vies_residuo":float(np.mean(residual)),"DP_residuos":float(np.std(residual,ddof=1)),
        "previsoes_negativas":int((pred<0).sum()),"busca_segundos":timing[name]["search_seconds"],
        "previsao_segundos":timing[name]["forecast_seconds"]})
    combined[name] = pred
    slug = {"MLP Regressor":"mlp","Random Forest":"random_forest","SARIMAX":"sarimax","Holt-Winters":"holt_winters"}[name]
    if name not in ["MLP Regressor","Random Forest"]:
        pd.DataFrame({"target_date":test.date,"origin_date":origin,"real":y_test,"prediction":pred,
            "residual":residual,"absolute_error":np.abs(residual)}).to_csv(OUT / slug / "walk_forward_predictions.csv",index=False)
    save_json(f"{slug}/config.json",selected[name])
comparison = pd.DataFrame(comparison_rows).sort_values("MAE_teste").reset_index(drop=True)
comparison.insert(0,"posicao",np.arange(1,5))
comparison.to_csv(OUT / "comparacao_modelos.csv",index=False)
combined.to_csv(OUT / "previsoes_comparadas.csv",index=False)
baseline_pred = np.array([np.median(data[TARGET].iloc[:cut+i]) for i in range(len(test))])
save_json("referencia_mediana.json",{"MAE":mean_absolute_error(y_test,baseline_pred),"definition":"Mediana expansiva dos gols observados antes de cada origem"})
subperiods = []
for j,ids in enumerate(np.array_split(np.arange(len(test)),4)):
    for name,pred in predictions.items():
        subperiods.append({"block":j+1,"start":test.date.iloc[ids[0]].strftime("%d/%m/%Y"),
            "end":test.date.iloc[ids[-1]].strftime("%d/%m/%Y"),"modelo":name,
            "MAE":mean_absolute_error(y_test[ids],pred[ids])})
pd.DataFrame(subperiods).to_csv(OUT / "mae_subperiodos.csv",index=False)
rng = np.random.default_rng(SEED)
winner = comparison.iloc[0].modelo
bootstrap_rows = []
for name in predictions:
    if name==winner: continue
    delta = np.abs(y_test-predictions[name])-np.abs(y_test-predictions[winner])
    draws = []
    for _ in range(2000):
        starts = rng.integers(0,len(test),size=int(np.ceil(len(test)/10)))
        ids = np.concatenate([(start+np.arange(10))%len(test) for start in starts])[:len(test)]
        draws.append(np.mean(delta[ids]))
    lo,hi = np.quantile(draws,[.025,.975])
    bootstrap_rows.append({"modelo":name,"referencia":winner,"diferenca_MAE":float(np.mean(delta)),
                           "IC95_inferior":lo,"IC95_superior":hi,"bloco":10,"repeticoes":2000})
pd.DataFrame(bootstrap_rows).to_csv(OUT / "incerteza_mae.csv",index=False)
display(comparison)
fig, ax = plt.subplots(figsize=(8,3.6))
bars = ax.barh(comparison.modelo[::-1],comparison.MAE_teste[::-1],color=[COLORS[n] for n in comparison.modelo[::-1]])
ax.bar_label(bars,fmt="%.4f",padding=5)
ax.set(xlabel="Erro absoluto médio (gols por partida)",title="MAE no teste walk-forward")
ax.set_xlim(0,comparison.MAE_teste.max()*1.15)
savefig("mae",fig)
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(test.date.iloc[-60:],y_test[-60:],color="#0d253d",label="Real",linewidth=1.7)
for name,pred in predictions.items(): ax.plot(test.date.iloc[-60:],pred[-60:],color=COLORS[name],label=name,linewidth=1,alpha=.85)
ax.set(ylabel="Gols",title="Últimas 60 previsões fora da amostra")
ax.legend(ncol=3,fontsize=8)
savefig("previsoes",fig)
log("Comparação por MAE concluída")

,posicao,modelo,MAE_teste,vies_residuo,DP_residuos,previsoes_negativas,busca_segundos,previsao_segundos
0,1,Holt-Winters,1.259261,-0.003819,1.606149,0,3.119305,0.010057
1,2,Random Forest,1.268689,-0.031933,1.599330,0,118.180896,75.967694
2,3,MLP Regressor,1.305420,0.128176,1.641230,0,189.275570,66.335190
3,4,SARIMAX,1.369759,-0.169138,1.686079,0,339.772599,75.659420


Comparação por MAE concluída


## 9. Resíduos, ACF e Ljung-Box
Os resíduos são real menos previsto. O teste é aplicado a erros genuinamente
fora da amostra, em atrasos 5, 10 e 20, como diagnóstico descritivo (`model_df=0`).
P-valor acima de 0,05 indica ausência de evidência de autocorrelação nesses
atrasos; não comprova independência nem baixa magnitude dos erros.

In [13]:
lb_rows = []
for name,pred in predictions.items():
    residual = y_test-pred
    lb = acorr_ljungbox(residual,lags=[5,10,20],model_df=0,return_df=True)
    for lag,row in lb.iterrows():
        lb_rows.append({"modelo":name,"lag":int(lag),"Q":float(row.lb_stat),
                        "p_valor":float(row.lb_pvalue),"rejeita_5pct":bool(row.lb_pvalue<.05)})
    fig, axes = plt.subplots(1,3,figsize=(11,3))
    axes[0].plot(test.date,residual,color=COLORS[name],linewidth=.9)
    axes[0].axhline(0,color="#0d253d",linewidth=.8)
    axes[0].set(ylabel="Real − previsto",title=f"Resíduos — {name}")
    plot_acf(residual,lags=20,ax=axes[1],zero=False,title="ACF dos resíduos")
    axes[2].hist(residual,bins=20,color=COLORS[name],alpha=.85)
    axes[2].axvline(np.mean(residual),color="#0d253d",linestyle="--")
    axes[2].set(title="Distribuição e viés",xlabel="Resíduo")
    slug={"MLP Regressor":"mlp","Random Forest":"rf","SARIMAX":"sarimax","Holt-Winters":"hw"}[name]
    savefig(f"residuos_{slug}",fig)
lb_table = pd.DataFrame(lb_rows)
lb_table.to_csv(OUT / "ljung_box.csv",index=False)
display(lb_table)

,modelo,lag,Q,p_valor,rejeita_5pct
0,Random Forest,5,7.029054,0.218488,False
1,Random Forest,10,11.616149,0.311567,False
2,Random Forest,20,17.704621,0.606859,False
3,MLP Regressor,5,9.200595,0.101326,False
4,MLP Regressor,10,13.619683,0.191056,False
5,MLP Regressor,20,20.322135,0.437948,False
6,SARIMAX,5,10.358760,0.065685,False
7,SARIMAX,10,15.492866,0.115099,False
8,SARIMAX,20,23.334039,0.272692,False
9,Holt-Winters,5,6.893969,0.228646,False


## 10. Importância das variáveis e interpretação
Permutação aplicada aos modelos ajustados somente no treino inicial e avaliados
no teste, sem incorporar seus alvos no ajuste. É uma análise explicativa de um
modelo fixo; não reproduz o reajuste do walk-forward e não participa da seleção.
O incremento no MAE é medido em gols. Variáveis correlacionadas podem dividir
importância. A interpretação não estabelece causalidade.

In [14]:
importance_tables = {}
for name in ["MLP Regressor","Random Forest"]:
    model = initial_models[name]
    imp = permutation_importance(model,X_test,y_test,scoring="neg_mean_absolute_error",
                                  n_repeats=20,random_state=SEED,n_jobs=2)
    table = pd.DataFrame({"feature":features,"increase_MAE":imp.importances_mean,
                          "std":imp.importances_std}).sort_values("increase_MAE",ascending=False)
    importance_tables[name] = table
    slug = "mlp" if name=="MLP Regressor" else "rf"
    table.to_csv(OUT / f"importancia_permutacao_{slug}.csv",index=False)
    fig, ax = plt.subplots(figsize=(9,4))
    top = table.head(12).iloc[::-1]
    ax.barh(top.feature,top.increase_MAE,xerr=top["std"],color=COLORS[name],alpha=.85)
    ax.axvline(0,color="#0d253d",linewidth=.8)
    ax.set(xlabel="Aumento do MAE após permutação (gols)",title=f"Importância por permutação — {name}")
    savefig(f"importancia_{slug}",fig)
    display(table.head(12))
rf = initial_models["Random Forest"]
native = pd.DataFrame({"feature":rf.named_steps["preprocess"].get_feature_names_out(),
    "importance":rf.named_steps["model"].feature_importances_}).sort_values("importance",ascending=False)
native.to_csv(OUT / "importancia_nativa_rf.csv",index=False)
param_names = list(sar_initial.param_names)
exog_names = list(sar_pre.get_feature_names_out())
names = [exog_names[int(n[1:])-1] if n.startswith("x") and n[1:].isdigit() else n for n in param_names]
sar_coefficients = pd.DataFrame({"parameter":names,"coefficient":sar_initial.params,
                                 "p_value":sar_initial.pvalues})
sar_coefficients.to_csv(OUT / "coeficientes_sarimax.csv",index=False)
display(sar_coefficients)
mlp_est = initial_models["MLP Regressor"].named_steps["model"]
mlp_details = {"input_dimensions":int(mlp_est.n_features_in_),
    "parameters_count":int(sum(w.size for w in mlp_est.coefs_)+sum(b.size for b in mlp_est.intercepts_)),
    "iterations":int(mlp_est.n_iter_),"loss":float(mlp_est.loss_),
    "training_mae":mean_absolute_error(y_train,initial_models["MLP Regressor"].predict(X_train)),
    "fixed_test_mae":mean_absolute_error(y_test,initial_models["MLP Regressor"].predict(X_test)),
    "selected":selected["MLP Regressor"]}
save_json("estudo_mlp.json",mlp_details)
if hasattr(mlp_est,"loss_curve_"):
    fig, ax = plt.subplots(figsize=(8,3))
    ax.plot(mlp_est.loss_curve_,color="#533afd")
    ax.set(xlabel="Iteração",ylabel="Função objetivo",title="Aprendizado do MLP no treino inicial")
    savefig("curva_mlp",fig)

,feature,increase_MAE,std
38,opponent,0.022018,0.013576
37,points_roll_mean_20,0.018761,0.006999
12,points_lag_2,0.013822,0.006361
9,points_lag_1,0.005134,0.003417
17,conceded_lag_5,0.004379,0.002897
7,goals_lag_1,0.003830,0.001644
36,conceded_roll_mean_20,0.003649,0.006746
26,goals_roll_mean_5,0.003110,0.001845
1,log_days_since_last_match,0.002268,0.003559
22,goals_roll_mean_3,0.002227,0.002060


,feature,increase_MAE,std
38,opponent,0.007831,0.017275
5,doy_cos,0.006819,0.002457
7,goals_lag_1,0.004116,0.002314
22,goals_roll_mean_3,0.002893,0.002884
35,goals_roll_std_20,0.002723,0.002012
10,goals_lag_2,0.002555,0.001440
17,conceded_lag_5,0.002473,0.001492
21,points_lag_10,0.002434,0.001509
4,doy_sin,0.001990,0.002975
31,goals_roll_std_10,0.001666,0.002120


,parameter,coefficient,p_value
0,intercept,0.000566,9.956887e-02
1,numeric__is_home,0.423609,1.199844e-10
2,numeric__log_days_since_last_match,0.087980,1.954226e-01
3,numeric__month_sin,-0.055630,4.186326e-01
4,numeric__month_cos,-0.018231,7.980641e-01
5,numeric__year,2.005519,5.420206e-03
6,categorical__competition_group_FIFA Confederat...,0.713839,7.555966e-02
7,categorical__competition_group_FIFA World Cup,0.204772,3.094590e-01
8,categorical__competition_group_International F...,0.262181,1.821235e-01
9,categorical__competition_group_Outras competições,0.005444,9.776811e-01


## 11. Verificações de integridade e reprodução
Conferem-se cronologia, quantidade de previsões, alinhamento das origens,
recomputação do MAE a partir dos CSVs e ausência de alvos contemporâneos nos
atributos. O ambiente e os hashes dos resultados são registrados para auditoria.

In [15]:
for name in predictions:
    assert np.isclose(comparison.set_index("modelo").loc[name,"MAE_teste"],
                      np.mean(np.abs(combined["real"]-combined[name])))
assert len(comparison)==4 and set(comparison.modelo)==set(COLORS)
assert (combined.origin_date < combined.target_date).all()
assert combined.target_date.is_unique and len(combined)==len(test)
versions = {package:importlib.metadata.version(package) for package in
            ["numpy","pandas","scipy","scikit-learn","statsmodels","matplotlib","nbformat","nbclient"]}
save_json("ambiente.json",{"python":platform.python_version(),"libraries":versions,
                          "seed":SEED,"threads":2,"dataset_sha256":quality["sha256"]})
save_json("verificacoes.json",{"chronology":True,"shared_origins":True,"shared_test":True,
    "no_current_targets_in_features":True,"mae_recomputed":True,"required_models":True,
    "predictions_per_model":len(test),"source_unchanged":hashlib.sha256((ROOT/"datasets/brazil.csv").read_bytes()).hexdigest()==quality["sha256"]})
save_json("manifesto.json",{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(OUT.rglob("*.csv"))})
display(comparison[["posicao","modelo","MAE_teste"]])
log("Análise concluída: modelos, diagnósticos, importâncias e verificações salvos")

,posicao,modelo,MAE_teste
0,1,Holt-Winters,1.259261
1,2,Random Forest,1.268689
2,3,MLP Regressor,1.305420
3,4,SARIMAX,1.369759


Análise concluída: modelos, diagnósticos, importâncias e verificações salvos
